In [1]:
import pandas as pd
import os

# ================= 配置区 =================
KG_FILE = "kg.csv"  # PrimeKG 原始文件路径

# 针对 PPMI (UPDRS-III + T1 MRI) 定制的侦察组
# 逻辑：
# 1. 核心病理：虽然表格没基因，但图谱必须有 PD 的生物学定义 (Dopamine, Synuclein)
# 2. 运动症状：对应 CSV 中的 upd23xx 列 (Tremor, Rigidity, Gait)
# 3. 神经解剖：为 T1 MRI 提供“脑区”节点 (Putamen, Caudate)
SEARCH_GROUPS = {
    "【核心-PD病理机制】": [
        "Parkinson", "Dopamine", "Synuclein", "Lewy", "Neurodegeneration"
    ],
    "【PPMI-运动症状 (对应UPDRS-III列)】": [
        # 震颤 (upd2315-17)
        "Tremor", "Resting tremor", "Action tremor", 
        # 强直 (upd2303)
        "Rigidity", "Stiffness", "Muscle rigidity",
        # 迟缓 (upd2304-09, 14)
        "Bradykinesia", "Hypokinesia", "Slowness",
        # 步态与姿势 (upd2310-13)
        "Gait", "Posture", "Instability", "Freezing", "Festinating",
        # 面部与言语 (upd2301-02)
        "Speech", "Dysarthria", "Facial", "Hypomimia"
    ],
    "【MRI-神经解剖骨架 (对应T1结构)】": [
        # 基底节回路 (PD 核心受损区)
        "Substantia nigra", "Striatum", "Putamen", "Caudate", "Globus pallidus", 
        # 运动控制回路
        "Thalamus", "Motor cortex", "Basal ganglia", "Cerebellum"
    ]
}

def main():
    print(f"🚀 [Step 1] 正在加载 {KG_FILE} 进行 PPMI 适配侦察...")
    
    try:
        # 只读取必要列以节省内存
        df = pd.read_csv(KG_FILE, usecols=['relation', 'x_name', 'x_type', 'y_name', 'y_type'], low_memory=False)
    except ValueError:
        df = pd.read_csv(KG_FILE, low_memory=False)
        
    print(f"✅ PrimeKG 加载完成，共 {len(df)} 条数据。\n")

    # 1. 关键词命中测试
    print("="*60)
    print("🔎 1. 关键词命中测试 (确认 UPDRS 和 MRI 特征能否映射到生物图谱)")
    print("="*60)
    
    # 剔除药物和暴露 (我们要找的是生物学机制)
    mask_bio = (~df['x_type'].isin(['drug', 'exposure'])) & (~df['y_type'].isin(['drug', 'exposure']))
    df_bio = df[mask_bio]

    found_keywords = set()

    for group, keywords in SEARCH_GROUPS.items():
        print(f"\n--- {group} ---")
        pattern = '|'.join(keywords)
        
        # 模糊匹配
        mask = df_bio['x_name'].astype(str).str.contains(pattern, case=False, na=False) | \
               df_bio['y_name'].astype(str).str.contains(pattern, case=False, na=False)
        subset = df_bio[mask]
        
        if len(subset) > 0:
            print(f"   ✅ 命中 {len(subset)} 条相关关系")
            # 提取具体的节点名，方便后续确认
            related_nodes = set(subset['x_name']) | set(subset['y_name'])
            # 简单的过滤，只看包含关键词的节点
            hits = [n for n in related_nodes if any(k.lower() in str(n).lower() for k in keywords)]
            # 打印前 8 个示例，多看几个确保准确
            print(f"   📌 典型节点示例: {hits[:8]}")
            found_keywords.update(keywords)
        else:
            print(f"   ❌ 警告：未找到相关节点！请检查拼写或同义词。")

    # 2. 黑名单侦察 (Hub Node Detection)
    print("\n" + "="*60)
    print("💣 2. 黑名单侦察 (找出连接数过高且无特异性的干扰节点)")
    print("="*60)
    print("   说明：请特别注意那些与 '脑' (Brain) 无关的通用解剖部位。")
    print("   (例如: 'testis', 'lung', 'kidney' 如果出现在 Top 榜单，下一步必须剔除！)")
    print("   保留 'Blood' 可能是危险的，除非你想研究血脑屏障，建议后续也列入黑名单。\n")

    # 统计所有节点的度 (Degree)
    all_nodes = pd.concat([df_bio['x_name'], df_bio['y_name']])
    node_counts = all_nodes.value_counts().head(30)

    print(f"{'Rank':<5} | {'Node Name':<40} | {'Count':<10}")
    print("-" * 60)
    for i, (node, count) in enumerate(node_counts.items()):
        print(f"{i+1:<5} | {node:<40} | {count:<10}")

    print("\n✅ [Step 1] 侦察完成！请根据上述结果：")
    print("   1. 检查 '典型节点示例' 里是否有奇怪的东西 (比如搜索 Facial 命中了 Facial cream)。")
    print("   2. 将 Rank 前 20 中与神经系统无关的器官 (如 Liver, Spleen) 复制到下一步的 BLACKLIST_NODES。")

if __name__ == "__main__":
    if os.path.exists(KG_FILE):
        main()
    else:
        print(f"❌ 找不到文件: {KG_FILE}")

🚀 [Step 1] 正在加载 kg.csv 进行 PPMI 适配侦察...
✅ PrimeKG 加载完成，共 8100498 条数据。

🔎 1. 关键词命中测试 (确认 UPDRS 和 MRI 特征能否映射到生物图谱)

--- 【核心-PD病理机制】 ---
   ✅ 命中 5866 条相关关系
   📌 典型节点示例: ['dopamine transport', 'neurodegeneration, childhood-onset, stress-induced, with variable ataxia and seizures', 'A8 dopaminergic cell group', 'somato-dendritic dopamine secretion', 'brain abnormalities, neurodegeneration, and dysosteosclerosis', 'negative regulation of dopamine metabolic process', 'negative regulation of adenylate cyclase-inhibiting dopamine receptor signaling pathway', 'dopamine uptake']

--- 【PPMI-运动症状 (对应UPDRS-III列)】 ---
   ✅ 命中 21536 条相关关系
   📌 典型节点示例: ['facial hypertrichosis (disease)', 'linear hypopigmentation and craniofacial asymmetry with acral, ocular and brain anomalies', 'nerve to stylohyoid from facial nerve', 'Facial flushing after alcohol intake', 'intellectual disability-severe speech delay-mild dysmorphism syndrome', 'Waddling gait', 'anterior facial vein', 'spondylometaphyseal dysplasia-bo

In [2]:
import pandas as pd
import os

# ================= 配置区 =================
# PPMI 文件列表
PPMI_FILES = ['control.csv', 'PD1.csv', 'prodromal.csv', 'swedd.csv']

# 定义症状组 (将细分的列归类为核心概念)
# 逻辑：只要组内任意一个列的值 > 0，就视为该病人有此症状
SYMPTOM_GROUPS = {
    "Speech (言语障碍)": ["code_upd2301_speech_problems"],
    "Facial (面部表情/面具脸)": ["code_upd2302_facial_expression"],
    
    # 强直：包括颈部和四肢
    "Rigidity (强直)": [
        "code_upd2303a_rigidity_neck", 
        "code_upd2303b_rigidity_rt_upper_extremity",
        "code_upd2303c_rigidity_left_upper_extremity", 
        "code_upd2303d_rigidity_rt_lower_extremity",
        "code_upd2303e_rigidity_left_lower_extremity"
    ],
    
    # 迟缓：包括指尖拍打、手部运动、轮替等
    "Bradykinesia (运动迟缓)": [
        "code_upd2304a_right_finger_tapping", "code_upd2304b_left_finger_tapping",
        "code_upd2305a_right_hand_movements", "code_upd2305b_left_hand_movements",
        "code_upd2306a_pron_sup_movement_right_hand", "code_upd2306b_pron_sup_movement_left_hand",
        "code_upd2307a_right_toe_tapping", "code_upd2307b_left_toe_tapping",
        "code_upd2308a_right_leg_agility", "code_upd2308b_left_leg_agility",
        "code_upd2314_body_bradykinesia"
    ],
    
    # 步态与姿势
    "Gait/Posture (步态/姿势)": [
        "code_upd2309_arising_from_chair",
        "code_upd2310_gait",
        "code_upd2311_freezing_of_gait",
        "code_upd2312_postural_stability",
        "code_upd2313_posture"
    ],
    
    # 震颤：包括静止性(Rest)和动作性(Postural/Kinetic)
    "Tremor (震颤 - 所有类型)": [
        "code_upd2315a_postural_tremor_of_right_hand", "code_upd2315b_postural_tremor_of_left_hand",
        "code_upd2316a_kinetic_tremor_of_right_hand", "code_upd2316b_kinetic_tremor_of_left_hand",
        "code_upd2317a_rest_tremor_amplitude_right_upper_extremity", 
        "code_upd2317b_rest_tremor_amplitude_left_upper_extremity",
        "code_upd2317c_rest_tremor_amplitude_right_lower_extremity",
        "code_upd2317d_rest_tremor_amplitude_left_lower_extremity",
        "code_upd2317e_rest_tremor_amplitude_lip_or_jaw"
    ]
}

def analyze_ppmi_prevalence():
    print("📊 正在统计 PPMI 临床数据中的运动症状出现频率...")
    
    # 1. 读取并合并所有数据
    df_list = []
    for fname in PPMI_FILES:
        if os.path.exists(fname):
            try:
                temp_df = pd.read_csv(fname)
                df_list.append(temp_df)
            except Exception as e:
                print(f"⚠️ 读取 {fname} 失败: {e}")
    
    if not df_list:
        print("❌ 没有读到任何数据！")
        return

    full_df = pd.concat(df_list, ignore_index=True)
    total_patients = len(full_df)
    
    stats = {k: 0 for k in SYMPTOM_GROUPS.keys()}
    
    # 2. 统计每个症状组的“患病”人数
    # 逻辑：UPDRS 评分 > 0 即视为有该症状
    for label, cols in SYMPTOM_GROUPS.items():
        # 找到实际存在的列 (防止列名拼写错误导致崩溃)
        valid_cols = [c for c in cols if c in full_df.columns]
        
        if valid_cols:
            # 检查每一行：如果这些列里有任意一个值 > 0，则 mask 为 True
            # fillna(0) 是为了防止空值报错，空值视为无症状
            has_symptom = (full_df[valid_cols].fillna(0).max(axis=1) > 0)
            stats[label] = has_symptom.sum()
        else:
            print(f"⚠️ 警告: 组 '{label}' 中的所有列在文件中都找不到！")

    # 3. 打印报告
    print(f"\n👥 总样本量 (合并后): {total_patients}")
    print("-" * 60)
    print(f"{'特征 (UPDRS-III)':<25} | {'出现人数':<10} | {'占比 (%)':<10}")
    print("-" * 60)
    
    sorted_stats = sorted(stats.items(), key=lambda x: x[1], reverse=True)
    
    for label, count in sorted_stats:
        ratio = (count / total_patients) * 100
        print(f"{label:<25} | {count:<10} | {ratio:.1f}%")
        
    print("-" * 60)
    print("💡 决策建议：")
    print("   - 震颤、强直、迟缓通常是 PD 的核心特征，占比应该很高。")
    print("   - 注意观察 Control 组是否也混入了一些轻微症状（这是正常的，老年人可能会有轻微震颤）。")
    print("   - 如果某项特征占比极低 (<5%)，在构建图谱时可以考虑不作为主要连接点。")

if __name__ == "__main__":
    analyze_ppmi_prevalence()

📊 正在统计 PPMI 临床数据中的运动症状出现频率...

👥 总样本量 (合并后): 804
------------------------------------------------------------
特征 (UPDRS-III)            | 出现人数       | 占比 (%)    
------------------------------------------------------------
Bradykinesia (运动迟缓)       | 649        | 80.7%
Rigidity (强直)             | 562        | 69.9%
Tremor (震颤 - 所有类型)        | 495        | 61.6%
Facial (面部表情/面具脸)         | 490        | 60.9%
Gait/Posture (步态/姿势)      | 452        | 56.2%
Speech (言语障碍)             | 293        | 36.4%
------------------------------------------------------------
💡 决策建议：
   - 震颤、强直、迟缓通常是 PD 的核心特征，占比应该很高。
   - 注意观察 Control 组是否也混入了一些轻微症状（这是正常的，老年人可能会有轻微震颤）。
   - 如果某项特征占比极低 (<5%)，在构建图谱时可以考虑不作为主要连接点。
